# 🤖 Notebook 3 — Modelling, Evaluation & SHAP Explainability

## What this notebook does
- Trains Random Forest and XGBoost classifiers
- Evaluates using AUC-ROC and Precision-Recall (not accuracy!)
- Compares both models systematically
- Uses SHAP values to explain what drives fraud predictions
- Quantifies business impact: money saved by deploying the model

**Input:** `data/processed/features_train.csv`, `data/processed/features_test.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              RocCurveDisplay, PrecisionRecallDisplay,
                              confusion_matrix, classification_report)
from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi':130, 'font.size':11})

train = pd.read_csv('../data/processed/features_train.csv')
test  = pd.read_csv('../data/processed/features_test.csv')

FEATURES = [c for c in train.columns if c != 'is_fraud']
X_train, y_train = train[FEATURES], train['is_fraud']
X_test,  y_test  = test[FEATURES],  test['is_fraud']

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 1. Train Models

In [ ]:
# ── Random Forest ────────────────────────────────────────────────────────
print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:,1]
rf_auc   = roc_auc_score(y_test, rf_proba)
rf_ap    = average_precision_score(y_test, rf_proba)
print(f'  AUC-ROC: {rf_auc:.4f} | Avg Precision: {rf_ap:.4f}')

# ── XGBoost ──────────────────────────────────────────────────────────────
print('Training XGBoost...')
scale_pos = (y_train==0).sum()/(y_train==1).sum()
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    eval_metric='auc', random_state=42, verbosity=0
)
xgb.fit(X_train, y_train)
xgb_proba = xgb.predict_proba(X_test)[:,1]
xgb_auc   = roc_auc_score(y_test, xgb_proba)
xgb_ap    = average_precision_score(y_test, xgb_proba)
print(f'  AUC-ROC: {xgb_auc:.4f} | Avg Precision: {xgb_ap:.4f}')

print(f'\n🏆 Winner: {"XGBoost" if xgb_auc > rf_auc else "Random Forest"}')

## 2. Model Comparison — ROC & Precision-Recall Curves

> **Why both curves?** ROC-AUC can be misleading with imbalanced datasets. Precision-Recall tells us how well the model performs specifically on the minority (fraud) class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(y_test, rf_proba,  name=f'Random Forest (AUC={rf_auc:.3f})',  ax=axes[0], color='#42A5F5')
RocCurveDisplay.from_predictions(y_test, xgb_proba, name=f'XGBoost (AUC={xgb_auc:.3f})',      ax=axes[0], color='#E53935')
axes[0].plot([0,1],[0,1],'k--',alpha=0.4,label='Random baseline')
axes[0].set_title('ROC Curve — Both Models')
axes[0].legend(loc='lower right')

PrecisionRecallDisplay.from_predictions(y_test, rf_proba,  name=f'Random Forest (AP={rf_ap:.3f})',  ax=axes[1], color='#42A5F5')
PrecisionRecallDisplay.from_predictions(y_test, xgb_proba, name=f'XGBoost (AP={xgb_ap:.3f})',      ax=axes[1], color='#E53935')
axes[1].set_title('Precision-Recall Curve — Both Models')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.savefig('../reports/figures/05_model_comparison.png', bbox_inches='tight')
plt.show()

## 3. Confusion Matrix & Classification Report — XGBoost

In [ ]:
THRESHOLD = 0.35  # Lower threshold to catch more fraud (business decision)
xgb_pred  = (xgb_proba >= THRESHOLD).astype(int)

cm = confusion_matrix(y_test, xgb_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
sns.heatmap(cm, annot=True, fmt=',', cmap='Blues', ax=axes[0],
            xticklabels=['Predicted Legit','Predicted Fraud'],
            yticklabels=['Actual Legit','Actual Fraud'])
axes[0].set_title(f'Confusion Matrix (threshold={THRESHOLD})')

# Classification report as heatmap
report = classification_report(y_test, xgb_pred,
                                target_names=['Legitimate','Fraud'],
                                output_dict=True)
report_df = pd.DataFrame(report).T.drop('support', axis=1).iloc[:2]
sns.heatmap(report_df, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.5, vmax=1.0, ax=axes[1], linewidths=0.5)
axes[1].set_title('Classification Report Heatmap')

plt.tight_layout()
plt.savefig('../reports/figures/06_confusion_matrix.png', bbox_inches='tight')
plt.show()
print(classification_report(y_test, xgb_pred, target_names=['Legitimate','Fraud']))

## 4. SHAP Explainability

> SHAP (SHapley Additive exPlanations) tells us *why* the model made each prediction. This is critical for regulated industries (banking, insurance) where black-box models are not acceptable.

In [ ]:
print('Computing SHAP values (this takes ~30 seconds)...')
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plt.sca(axes[0])
shap.summary_plot(shap_values, X_test, plot_type='bar',
                  show=False, max_display=12,
                  color='#E53935')
axes[0].set_title('Top 12 Features by Mean |SHAP| Value')

plt.sca(axes[1])
shap.summary_plot(shap_values, X_test, show=False, max_display=12)
axes[1].set_title('SHAP Value Distribution per Feature')

plt.tight_layout()
plt.savefig('../reports/figures/07_shap_summary.png', bbox_inches='tight')
plt.show()
print('\n✓ SHAP analysis complete')

## 5. 💰 Business Impact Analysis

> This section translates model performance into financial value — the language every stakeholder understands.

In [ ]:
# ── Business assumptions ─────────────────────────────────────────────────
AVG_FRAUD_AMOUNT      = test.loc[test['is_fraud']==1, 'amount'].mean()
COST_FALSE_POSITIVE   = 15   # £ cost of manual review per flagged transaction
COST_MISSED_FRAUD     = AVG_FRAUD_AMOUNT * 1.2  # fraud amount + chargeback cost

tn, fp, fn, tp = cm.ravel()

fraud_caught_value  = tp * AVG_FRAUD_AMOUNT
false_alert_cost    = fp * COST_FALSE_POSITIVE
missed_fraud_cost   = fn * COST_MISSED_FRAUD
net_value           = fraud_caught_value - false_alert_cost - missed_fraud_cost

print('='*52)
print('        BUSINESS IMPACT SUMMARY (Test Set)        ')
print('='*52)
print(f'  Fraud transactions caught (True Positives):  {tp:>6,}')
print(f'  Fraud transactions missed (False Negatives): {fn:>6,}')
print(f'  Legitimate flagged wrongly (False Positives):{fp:>6,}')
print(f'')
print(f'  Avg fraudulent transaction value:            £{AVG_FRAUD_AMOUNT:>7.0f}')
print(f'  Total fraud value caught:                    £{fraud_caught_value:>7,.0f}')
print(f'  Cost of false alerts (manual review):       -£{false_alert_cost:>7,.0f}')
print(f'  Cost of missed fraud:                       -£{missed_fraud_cost:>7,.0f}')
print('='*52)
print(f'  NET VALUE ADDED BY MODEL:                    £{net_value:>7,.0f}')
print('='*52)
print(f'\n  Model AUC-ROC: {xgb_auc:.4f}')
print(f'  Precision on fraud class: {tp/(tp+fp):.3f}')
print(f'  Recall on fraud class:    {tp/(tp+fn):.3f}')